# Colab bootstrap

Template cells every per-experiment notebook starts from. Runtime: `Runtime > Change runtime type > GPU`.

This mirrors the Docker dev environment's dependencies (see `docker/requirements/colab.txt`) but does **not** reinstall torch — Colab already ships a CUDA-matched build.

In [ ]:
# 1. Mount Drive (for persistent checkpoint/metric storage across ephemeral runtimes)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Authenticate with the Hugging Face Hub using the token stored in
# Colab's Secrets (key icon in the left sidebar) -- avoids the "sending
# unauthenticated requests" warning and gets higher rate limits / faster
# model + dataset downloads. Requires a secret named HF_TOKEN with
# "Notebook access" enabled for this notebook (toggle in the Secrets panel).
# huggingface_hub/transformers/datasets all pick up this env var
# automatically, including in the `!python ...` subprocess cells below.
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# 3. Get the code from the public GitHub repo
!git clone https://github.com/jtylerleake/LoRA-Experiments.git /content/lora_experiments
%cd /content/lora_experiments

In [ ]:
# 4. Install deps on top of Colab's preinstalled torch, then install the package itself
!pip install -q -r docker/requirements/colab.txt
!pip install -q -e .
# Colab preinstalls an old torchao (quantization library we don't use -- our
# quantization backend is bitsandbytes). peft's LoRA module dispatcher probes
# every optional backend including torchao, and its version check raises
# instead of skipping when torchao is present but too old for that peft
# release, crashing get_peft_model() entirely. Removing it lets that check
# report "not available" and fall through to the default LoRA path.
!pip uninstall -y -q torchao

In [ ]:
# 5. Run an experiment (real GPU run - swap --dry-run for the CPU sanity check)
!python scripts/run_experiment.py \
    --config src/config/experiment_1_rank_ablation.yaml \
    --device cuda \
    --output-root /content/drive/MyDrive/lora_experiments_outputs